In [1]:
%pip install stripe
%pip install python-docx

Note: you may need to restart the kernel to use updated packages.
  Using cached python_docx-1.2.0-py3-none-any.whl.metadata (2.0 kB)
Using cached python_docx-1.2.0-py3-none-any.whl (252 kB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Final
import os
import sqlite3
import gradio as gr
import stripe
import pdfplumber
import uuid
import json
import threading

from datetime import datetime
from new_functions import (
    extract_resume_text,
    sanitize_input,
    prompt_creator,
    get_resume_response,
    process_resume,
    export_resume,
    cover_letter_prompt_creator,
    get_cover_response,
    save_cover_letter,
    is_posting_recent,
    template_detector,
    mentioned_on_socials,
    detect_urgency_language
)

# for handling the api stuff
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse

# ...and because fastapi is asynchronous, we need a server:
import uvicorn

# get new SQLite connection for each request
def get_db_connection():
    """ Function to let FastAPI handle multiple requests to prevent db lockup"""
    conn = sqlite3.connect("resumewhip.db", check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn

# Stripe setup
stripe.api_key = os.getenv("STRIPE_SECRET_KEY")
WEBHOOK_SECRET = os.getenv("STRIPE_WEBHOOK_SECRET")
PRICE_ID = os.getenv("PRICE_ID")

# Fastapi for the webhooks
fastapi_app = FastAPI()

# Simple in-memory storage - but sync with database
user_sessions = {}  # Maps session IDs to user data
FREE_CREDITS = 3

def init_database():
    """Initialize SQLite database with required tables"""
    conn = sqlite3.connect('resumewhip.db')
    cursor = conn.cursor()
    cursor.execute('''
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                user_id TEXT UNIQUE NOT NULL,
                email TEXT,
                subscription_status TEXT DEFAULT 'free',
                credits_remaining INTEGER DEFAULT 3,
                stripe_customer_id TEXT,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                last_payment_date TIMESTAMP
            )
        ''')
        
    conn.commit()
    conn.close()

def get_user_id():
    """Generate a persistent session-based user ID"""
    # Use Gradio's session state if available, otherwise create persistent ID
    if hasattr(gr, 'current_user_id') and gr.current_user_id:
        return gr.current_user_id
    
    # Create new user ID and store in session
    user_id = str(uuid.uuid4())
    gr.current_user_id = user_id
    return user_id

def get_user_credits(user_id):
    """ Retrieve user credits from database"""
    conn = get_db_connection()
    cursor = conn.cursor()

    # check to see if user already exists
    cursor.execute("SELECT credits_remaining, subscription_status FROM users WHERE user_id = ?", (user_id,))
    row = cursor.fetchone()
        
    # if they are new / don't already exist
    if row is None:
            default_credits = 3
            cursor.execute(
                "INSERT INTO users (user_id, credits_remaining, subscription_status) VALUES (?, ?, ?)", 
                (user_id, default_credits, 'free')
            )
            conn.commit()
            conn.close()
            return default_credits, 'free'

    # since they alreay exist, return their current credits  
    conn.close()
    return row["credits_remaining"], row["subscription_status"]

def update_user_credits(user_id, credits, subscription_status='free'):
    """ update user credits """
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("UPDATE users SET credits_remaining = ? WHERE user_id = ?", (credits, user_id))
    conn.commit()
    conn.close()

def get_user_id_by_customer_id(customer_id):
    """Get user_id from database using customer_id"""
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT user_id FROM users WHERE stripe_customer_id = ?", (customer_id,))
    row = cursor.fetchone()
    conn.close()
    return row["user_id"] if row else None

def get_stripe_customer_id_from_db(user_id):
    """Get the Stripe customer ID from SQLite db"""
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT stripe_customer_id FROM users WHERE user_id = ?",
        (user_id,)
    )
    row = cursor.fetchone()
    conn.close()
    return row["stripe_customer_id"] if row else None

def store_stripe_customer_id(user_id, customer_id):
    """Store Stripe customer ID in SQLite when they first subscribe"""
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE users SET stripe_customer_id = ? WHERE user_id = ?",
        (customer_id, user_id))
    conn.commit()
    conn.close()

def create_checkout_session():
    try:
        user_id = get_user_id()
        session = stripe.checkout.Session.create(
            client_reference_id=user_id,
            payment_method_types=['card'],
            line_items=[{
                'price': PRICE_ID,
                'quantity': 1,
            }],
            mode='subscription',
            success_url="https://www.resumewhip.com/success",
            cancel_url="https://resumewhip.com/cancel"
        )
        return session.url
    except Exception as e:
        print(f"Error creating checkout session: {e}")
        return f"Error creating checkout session: {e}"

def check_payment_status(user_id):
    """Function to check if the user has paid for the service using Stripe's API"""
    try:
        # check db for stored id
        customer_id = get_stripe_customer_id_from_db(user_id)
        if not customer_id:
            return False
        
        # verify subscription with Stripe
        subscriptions = stripe.Subscription.list(
            customer=customer_id,
            status='active', 
            limit=1
        )

        # see if they already have a subscription
        if subscriptions.data:
            subscription = subscriptions.data[0]
            # make sure that subscription is for my product
            if subscription.items.data[0].price.id == PRICE_ID:
                return True
        
        return False
    
    except stripe.error.StripeError as e:
        print(f"Stripe error in checking subscription: {e}")
        return False
    
    except Exception as e:
        print(f"Error in checking your payment status: {e}")
        return False

def grant_unlimited_access(user_id):
    """If they've paid, they get full access"""
    conn = get_db_connection()
    cursor = conn.cursor()

    # make sure they exist
    cursor.execute("SELECT 1 FROM users WHERE user_id = ?", (user_id,))
    if cursor.fetchone() is None:
        cursor.execute(
            "INSERT INTO users (user_id, credits_remaining, subscription_status) VALUES (?, ?, ?)",
            (user_id, -1, 'paid')
        )

    else:
        cursor.execute(
            "UPDATE users SET credits_remaining = -1, subscription_status = 'paid' WHERE user_id = ?",
            (user_id,)
        )

    conn.commit()
    conn.close()

def revoke_unlimited_access(user_id):
    """Revoke unlimited access and set user to free; create if missing"""
    conn = get_db_connection()
    cursor = conn.cursor()

    # see if they exist
    cursor.execute("SELECT 1 FROM users WHERE user_id = ?", (user_id,))
    if cursor.fetchone() is None:
        cursor.execute(
            "INSERT INTO users (user_id, credits_remaining, subscription_status) VALUES (?, ?, ?)",
            (user_id, 3, 'free')
        )
    else:
        cursor.execute(
        "UPDATE users SET subscription_status = 'free' WHERE user_id = ?",
        (user_id,) 
    )
    conn.commit()
    conn.close()

@fastapi_app.post("/webhook")
async def stripe_webhook(request: Request):
    """Stripe webhook endpoint for subscription and payment events."""
    try:
        # Get raw body and Stripe signature
        payload = await request.body()
        sig_header = request.headers.get("stripe-signature")

        if not sig_header or not WEBHOOK_SECRET:
            print("🚨 Missing Stripe signature or webhook secret.")
            raise HTTPException(status_code=400, detail="Missing signature or webhook secret")

        # Verify Stripe signature
        event = stripe.Webhook.construct_event(payload, sig_header, WEBHOOK_SECRET)

        # ✅ Handle successful payment
        if event["type"] == "checkout.session.completed":
            session = event["data"]["object"]
            user_id = session.get("client_reference_id")
            customer_id = session.get("customer")

            if user_id and customer_id:
                store_stripe_customer_id(user_id, customer_id)
                grant_unlimited_access(user_id)
                print(f"✅ Granted unlimited access to user: {user_id}")

        # ❌ Handle subscription cancellations
        elif event["type"] == "customer.subscription.deleted":
            subscription = event["data"]["object"]
            customer_id = subscription["customer"]
            user_id = get_user_id_by_customer_id(customer_id)

            if user_id:
                revoke_unlimited_access(user_id)
                print(f"❌ Revoked access for user: {user_id}")

        else:
            print(f"ℹ️ Unhandled event type: {event['type']}")

        # Return success so Stripe stops retrying
        return JSONResponse(content={"status": "success"}, status_code=200)

    except stripe.error.SignatureVerificationError as e:
        print(f"🚨 Invalid signature: {e}")
        raise HTTPException(status_code=400, detail="Invalid signature")

    except Exception as e:
        print(f"🚨 Webhook error: {e}")
        return JSONResponse(content={"status": "error", "message": str(e)}, status_code=400)

def run_resume_with_credits(resume_file, job_input):
    """Handle resume processing with credit system - fixed version"""
    if not resume_file or not job_input.strip():
        return ("⚠️ Please upload your resume and paste the job description they've provided.", "", "", "Free resumes left: -")
    
    user_id = get_user_id()
    
    # Check subscription status from database
    credits, subscription_status = get_user_credits(user_id)
    
    # Double-check with Stripe for premium users
    if subscription_status == 'premium' or check_payment_status(user_id):
        credits = float("inf")
        if subscription_status != 'premium':
            update_user_credits(user_id, float("inf"), 'premium')
    
    # Generate unique resume ID for tracking
    resume_name = getattr(resume_file, "name", "unknown")
    new_resume_id = f"{resume_name}_{hash(job_input)}"
    
    # Check if this is a new resume (consumes credit) or edit of existing one
    current_resume = user_sessions.get(f"{user_id}_current_resume")
    
    if current_resume != new_resume_id:
        if credits != float("inf"):
            if credits <= 0:
                checkout_url = create_checkout_session()
                return (
                    f"⏰ You've used your 3 free resumes! Ready for unlimited access? [Subscribe here]({checkout_url}) for just $5.99/month!",
                    "", "", "Free resumes left: 0"
                )
            credits -= 1
            update_user_credits(user_id, credits)
        
        user_sessions[f"{user_id}_current_resume"] = new_resume_id
    
    # Process resume
    try:
        result = process_resume(resume_file, job_input)
        credits_display = '∞' if credits == float('inf') else str(credits)
        return (*result, f"Free resumes left: {credits_display}")
    except Exception as e:
        print(f"Resume processing error: {e}")
        return (f"Error processing resume: {e}", "", "", f"Free resumes left: {credits}")

def generate_cover_letter(resume_file, job_input):
    """Generate cover letter from resume and job description"""
    if not resume_file or not job_input.strip():
        return "⚠️ Please check that you've uploaded a resume and pasted in the job description."
    
    try:
        # Extract text from resume file
        resume_txt = extract_resume_text(resume_file)
        if not resume_txt:
            return "⚠️ For some reason, we could not extract text from the resume file. Please try again."
        
        prompt = cover_letter_prompt_creator(resume_txt, job_input)
        return get_cover_response(prompt)
    except Exception as e:
        return f"🚩 Unexpected error in generating cover letter: {e}"

def validate_job_posting(job_input_text, posting_date, company, job_title):
    """Validate job posting legitimacy"""
    # Fixed logic - check if fields are empty (not inverted)
    if not company.strip():
        return "⚠️ Please enter a company name to validate the job posting."
    
    if not job_input_text.strip():
        return "⚠️ Please paste the job description to validate."
    
    if not posting_date.strip():
        return "⚠️ Please provide a job posting date (YYYY-MM-DD format)."
    
    try:
        # Validate job posting
        recent = is_posting_recent(posting_date)
        template_flag = template_detector(job_input_text)
        urgency_flag = detect_urgency_language(job_input_text)
        social_links = mentioned_on_socials(company, job_title or "")
        
        report = "### 🕐 Posting Date Check:\n"
        report += "✅ Yup, the job looks recent (posted within 60 days).\nIn this market, jobs don't stay open for more than that." if recent else "⚠️ Warning: Job may be outdated (older than 60 days).\nCould be they're just harvesting candidates."
        
        report += "\n### 🤖 Template Language Check:\n"
        report += "⚠️ Generic/template language detected - proceed with caution.\nCould be just a cattle call for info to keep on file." if template_flag else "✅ Posting appears specific and legitimate - as if an actual person wrote it and they have an actual need.\n"
        
        report += "\n### ⚡ Urgency Language Check:\n"
        report += "⚠️ Urgency language detected - be cautious of scams.\nCheck the post for poor grammar, unrealistic salary / work expectations, and that 'too good to be true' feel." if urgency_flag else "✅ No suspicious urgency language found.\nSeems like a real job posting."
        
        report += "\n### 🔍 Verify the Job / Company on Social Media:\n"
        report += f"- [Search on X/Twitter]({social_links['x']})\n"
        report += f"- [Search on LinkedIn]({social_links['linkedin']})\n"
        
        return report
        
    except Exception as e:
        return f"🚩 Error validating job posting: {e}"

# Admin for granting access
def admin_grant_access(user_email_or_id):
    """ Function to grant unlimited access """
    user_id = get_user_id()
    grant_unlimited_access(user_id)
    return f"Granted unlimited access to user: {user_id}"

# Sticky buy button + banner (add before "with gr.Blocks()")
sticky_buy_button = """
<div style="position:fixed; top:10px; right:20px; z-index:9999; text-align:center;">
    <a href="https://buy.stripe.com/cNi9ASgWl6C614l3Ja1Jm00" target="_blank" 
       style="background-color:#ff7f50; color:white; padding:12px 20px; 
              text-decoration:none; border-radius:8px; font-size:1em; 
              font-weight:bold; box-shadow:0px 2px 6px rgba(0,0,0,0.2);">
        ♾️ Unlimited Resume Optimization — $5.99/mo
    </a>
    <div style="font-size:0.8em; color:#333; margin-top:4px;">
        Trusted by 200+ job seekers
    </div>
</div>
"""

# Custom CSS for colored tabs and better styling
# Replace your existing custom_css variable with this enhanced version:

custom_css = """
<style>
        /* === HERO SECTION === */
        .hero-section {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 60px 0;
            margin-bottom: 40px;
            text-align: center;
            position: relative;
            overflow: hidden;
        }

        .hero-section::before {
            content: '';
            position: absolute;
            top: 0;
            left: 0;
            right: 0;
            bottom: 0;
            background: radial-gradient(circle, rgba(255,255,255,0.1) 1px, transparent 1px);
            background-size: 50px 50px;
            animation: float 20s ease-in-out infinite;
            pointer-events: none;
        }

        @keyframes float {
            0%, 100% { transform: translateY(0px); }
            50% { transform: translateY(-10px); }
        }

        .hero-title {
            font-size: 3.5em;
            font-weight: 900;
            text-shadow: 3px 3px 6px rgba(0,0,0,0.3);
            margin-bottom: 20px;
            position: relative;
            z-index: 2;
        }

        .hero-subtitle {
            font-size: 2em;
            color: #ff7f50;
            font-weight: 700;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.3);
            margin-bottom: 15px;
            position: relative;
            z-index: 2;
        }

        .hero-offer {
            font-size: 1.8em;
            color: #ff7f50;
            font-weight: 600;
            margin-bottom: 10px;
            position: relative;
            z-index: 2;
        }

        .hero-description {
            font-size: 1.3em;
            font-weight: 500;
            position: relative;
            z-index: 2;
            margin-bottom: 0;
        }

        /* === SIDEBAR === */
        .sidebar {
            background: linear-gradient(135deg, #f8f9fa 0%, #e9ecef 100%);
            min-height: 100vh;
            padding: 30px 20px;
            border-right: 3px solid #dee2e6;
        }

        .btn-upgrade {
            background: linear-gradient(135deg, #635BFF 0%, #5a67d8 100%);
            color: white;
            border: none;
            padding: 15px 25px;
            border-radius: 12px;
            font-size: 1.1em;
            font-weight: 700;
            text-decoration: none;
            display: inline-block;
            transition: all 0.3s ease;
            box-shadow: 0 6px 20px rgba(99, 91, 255, 0.3);
        }

        .btn-upgrade:hover {
            transform: translateY(-3px);
            box-shadow: 0 10px 30px rgba(99, 91, 255, 0.4);
            color: white;
            text-decoration: none;
        }

        .privacy-note {
            text-align: center;
            margin-top: 30px;
            padding: 20px;
            background: rgba(40, 167, 69, 0.1);
            border-radius: 10px;
            border: 2px solid rgba(40, 167, 69, 0.2);
        }

        /* === MAIN CONTENT === */
        .main-content {
            padding: 30px;
        }

        .form-control {
            border: 2px solid #dee2e6;
            border-radius: 8px;
            padding: 12px 15px;
            font-size: 1em;
            transition: all 0.3s ease;
        }

        .form-control:focus {
            border-color: #667eea;
            box-shadow: 0 0 0 0.2rem rgba(102, 126, 234, 0.25);
        }

        .credits-counter {
            text-align: center;
            padding: 15px;
            background: linear-gradient(135deg, #28a745 0%, #20c997 100%);
            color: white;
            border-radius: 10px;
            font-weight: 600;
            font-size: 1.2em;
        }

        /* === SIMPLE TOOLS HEADER === */
        .tools-header {
            text-align: center;
            margin: 40px 0 30px 0;
            padding: 20px;
        }

        .tools-header h2 {
            font-size: 2.2em;
            color: #495057;
            font-weight: 700;
            margin-bottom: 15px;
        }

        .tools-header p {
            font-size: 1.1em;
            color: #6c757d;
            margin-bottom: 0;
        }

        /* === MAIN TOOL BUTTONS (These replace the tabs) === */
        .tool-buttons {
            display: flex;
            justify-content: center;
            gap: 20px;
            flex-wrap: wrap;
            margin: 30px 0;
        }

        .tool-button {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border: none;
            font-weight: 800;
            font-size: 1.4em;
            border-radius: 12px;
            padding: 25px 40px;
            transition: all 0.4s ease;
            text-transform: uppercase;
            letter-spacing: 1px;
            min-width: 240px;
            cursor: pointer;
            box-shadow: 0 6px 20px rgba(102, 126, 234, 0.3);
            position: relative;
        }

        .tool-button:hover {
            transform: translateY(-6px) scale(1.03);
            box-shadow: 0 15px 35px rgba(102, 126, 234, 0.5);
            background: linear-gradient(135deg, #5a67d8 0%, #667eea 100%);
        }

        .tool-button.active {
            background: linear-gradient(135deg, #ff7f50 0%, #ff6b35 100%);
            box-shadow: 0 8px 25px rgba(255, 127, 80, 0.6);
            transform: translateY(-3px);
        }

        .tool-button:not(.active) {
            animation: subtle-pulse 4s ease-in-out infinite;
        }

        @keyframes subtle-pulse {
            0%, 100% { box-shadow: 0 6px 20px rgba(102, 126, 234, 0.3); }
            50% { box-shadow: 0 10px 30px rgba(102, 126, 234, 0.4); }
        }

        /* === TOOL CONTENT AREAS === */
        .tool-content {
            display: none;
            background: #f8f9fa;
            border: 2px solid #dee2e6;
            border-radius: 15px;
            padding: 30px;
            margin-top: 30px;
        }

        .tool-content.active {
            display: block;
            animation: fadeIn 0.5s ease-in-out;
        }

        @keyframes fadeIn {
            from { opacity: 0; transform: translateY(20px); }
            to { opacity: 1; transform: translateY(0); }
        }

        .tool-content h3 {
            color: #495057;
            font-weight: 700;
            margin-bottom: 20px;
            text-align: center;
            font-size: 1.8em;
        }

        /* === ACTION BUTTONS === */
        .whip-button {
            background: linear-gradient(135deg, #e74c3c 0%, #c0392b 100%);
            color: white;
            font-size: 1.3em;
            font-weight: 700;
            padding: 20px 40px;
            border-radius: 12px;
            border: none;
            transition: all 0.3s ease;
            text-transform: uppercase;
            letter-spacing: 0.5px;
            box-shadow: 0 6px 15px rgba(231, 76, 60, 0.3);
            width: 100%;
            margin-top: 20px;
        }

        .whip-button:hover {
            transform: translateY(-4px);
            box-shadow: 0 10px 30px rgba(231, 76, 60, 0.5);
        }

        /* === RESULTS SECTIONS === */
        .results-section {
            background: white;
            border: 2px solid #dee2e6;
            border-radius: 12px;
            padding: 25px;
            margin-top: 20px;
        }

        .results-section h4 {
            color: #495057;
            font-weight: 700;
            margin-bottom: 15px;
            border-bottom: 2px solid #dee2e6;
            padding-bottom: 10px;
        }

        /* === RESPONSIVE === */
        @media (max-width: 768px) {
            .hero-title { font-size: 2.5em; }
            .hero-subtitle { font-size: 1.5em; }
            .tool-button {
                font-size: 1.2em;
                padding: 20px 30px;
                min-width: 200px;
                margin: 8px;
            }
            .tools-header h2 { font-size: 1.8em; }
        }
    </style>
"""

# Create Gradio interface
with gr.Blocks(title="ResumeWhip - AI Resume Optimizer | ATS-Friendly Resume Builder", theme=gr.themes.Soft(), css=custom_css) as app:
    
    gr.HTML("<head><link rel='icon' href='favicon.png' type='image/png'></head>")
    
    # Header
    gr.Markdown("""
    <h1 style='text-align:center; color:#1e90ff;'>🏎️💨 Welcome To ResumeWhip!</h1>
    <h2 style='text-align:center; color:#dd1eff;'>The AI-Powered Resume Optimizer Meant to Whip ATS Systems</h2>
    <h2 style='text-align:center; color:#dd1eff;'>Get Your First 3 Resumes Free!</h2> 
    <h3 style='text-align:center;'>Powerful. Simple. Just Validate → Upload → Optimize → Apply!</h3>          
    """)

    with gr.Row():
        # Sidebar
        with gr.Column(scale=1):
            with gr.Accordion("🦮 How To Use", open=False):
                gr.Markdown("""
                         1.) Crank Your Existing Resume Up To 11 - list every single skill and experience you have
                             (this is how our AI writes your resume and scores your chances);  
                         2.) Follow the Prompts To Load the Requested Info;  
                         3.) Choose Your Tool (you don't have to use all 3);  
                         4.) Proofread / Edit the Results Using the 'Copy/Pastes' below;  
                         5.) Download Your File as PDF; and  
                         6.) Apply!
                            """)
                
            with gr.Accordion("📚 Formatting Tips", open=False):
                gr.Code("""
If you're not happy with the default resume 
format, you can make adjustments using the 
simple copy and pastes below:
                        
Want To Change Fonts:
# = Biggest  
## = Smaller  
### = Smallest
<b>text</b> = Bold  
<i>text</i> = Italic  
<u>text</u> = Underline
(⬆️ Can Be Combined As Needed)
                        
Making A List?
- = Bullet Point  
1. = Numbered List

Want To Link A Website?
[Your Website](https://www.yourwebsite.com)

Too "clumpy?" Break things up into separate lines
with two spaces where you want the line break
(after a period, for example). 
                        
Or, if you want to create a section break with
a line, just enter three dashes (---) where you 
want that break to happen (eg: between 'Summary' 
and 'Experience').

"But my resume cuts off where I don't want it to!"
Simple - just force a new page by copy/pasting this entire 
line (below), and put it where you want one page to end
and the next to begin:
<div style="page-break-after: always; break-after: page;"></div>
                """, language="markdown")
                
            with gr.Accordion("❓ Frequently Asked Questions", open=False):
                gr.Markdown("""
        ### Common Questions About Resume Optimization
        
        **Q: What is an ATS?**
        A: "Applicant Tracking System." It's an automated filter Recruiters use to sift through the deluge of resumes they receive for open job positions.
                            
        **Q: How does your Resume Optimizer and Cover Letter Writer work?**  
        A: Our AI pits your resume up against job descriptions and tailors its content, keywords, and formatting to match what ATS systems and recruiters look for.
        
        **Q: What makes ResumeWhip any different than the other resume optimizers?**
        A: Its code is written by job seekers with fellow jobseekers in mind. In double-blind studies, it consistently outperforms the competition (eg: LinkedIn's AI Resume Optimizer).
                                       
        **Q: Do optimized resumes really get more interviews?**  
        A1: Yes - ATS-optimized, job-specific resumes typically see 3-5x higher response rates than generic versions. AI ensures you never miss the keywords ATS systems use to qualify your resume.
        
        **Q: What file formats work best with the resume optimizer?**  
        A: We support PDF(.pdf), Word (.docx), Markdown (.md), and text (.txt) files.
        
        **Q: How many resumes can I optimize for free?**  
        A: You get 3 free resume optimizations to try our service, then unlimited access for $5.99/month.

        **Q: Why is the format of the resume I download look so plain?**
        A: That's by design - ATS systems don't like a lot of formatting. (Tables and multiple columns? Nightmares for them.)
        """)
                
            # Subscribe button
            gr.HTML("""
            <div style="text-align:center; margin:20px 0;">
                <a href="https://buy.stripe.com/cNi9ASgWl6C614l3Ja1Jm00" 
                   target="_blank" 
                   style="background-color:#635BFF; color:white; padding:15px 25px; 
                          text-decoration:none; border-radius:8px; font-size:1.1em; 
                          font-weight:bold; display:inline-block;">
                    ♾️ Get Unlimited AI-Powered Resume Optimizaton for Just  $5.99/Month!
                </a>
            </div>
            """)
            
            gr.Markdown("### 🛡️ We will never sell your data. Ever.")

            gr.Markdown("### 🔥 Share the love - help friends skip the job search struggle!")

            gr.HTML("""
                <div style="display:flex; flex-direction:column; gap:20px; margin-top:20px;">
                    
                    <!-- Clean logo-based sharing -->
                    <div style="display:flex; flex-direction:row; gap:20px; justify-content:center; align-items:center;">
                        
                        <!-- LinkedIn -->
                        <a href="https://www.linkedin.com/sharing/share-offsite/?url=https://resumewhip.com&summary=Finally%20found%20a%20resume%20tool%20that%20actually%20understands%20ATS%20systems%20🎯%20ResumeWhip%20uses%20AI%20to%20optimize%20resumes%20for%20each%20job%20posting.%20Getting%20way%20more%20interviews%20than%20generic%20resumes!" 
                           target="_blank" 
                           style="text-decoration: none; position: relative; display: inline-block;"
                           onmouseover="showTooltip(this, 'Help your network with the AI resume optimizer that outperforms LinkedIn\'s own tools!')"
                           onmouseout="hideTooltip(this)">
                            <img src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" 
                                 style="width:40px; height:40px; transition: transform 0.3s ease, filter 0.3s ease;"
                                 onmouseover="this.style.transform='scale(1.2)'; this.style.filter='brightness(1.2)';"
                                 onmouseout="this.style.transform='scale(1)'; this.style.filter='brightness(1)';">
                        </a>
                        
                        <!-- X/Twitter -->
                        <a href="https://x.com/intent/post?url=https://resumewhip.com&text=Just%20boosted%20my%20resume%27s%20ATS%20score%20by%2040%25%20with%20AI!%20🎯%20Finally%20getting%20callbacks%20instead%20of%20radio%20silence.%20ResumeWhip%20tailors%20resumes%20to%20each%20job%20-%203%20free%20tries!" 
                           target="_blank"
                           style="text-decoration: none; position: relative; display: inline-block;"
                           onmouseover="showTooltip(this, 'Tweet about AI resume success - your followers will thank you!')"
                           onmouseout="hideTooltip(this)">
                            <img src="https://upload.wikimedia.org/wikipedia/commons/c/ce/X_logo_2023.svg" 
                                 style="width:40px; height:40px; transition: transform 0.3s ease, filter 0.3s ease;"
                                 onmouseover="this.style.transform='scale(1.2)'; this.style.filter='brightness(1.2)';"
                                 onmouseout="this.style.transform='scale(1)'; this.style.filter='brightness(1)';">
                        </a>
                        
                        <!-- Facebook -->
                        <a href="https://www.facebook.com/sharer/sharer.php?u=https://resumewhip.com&quote=Stop%20sending%20resumes%20into%20the%20void!%20🎯%20This%20AI%20tool%20actually%20gets%20you%20past%20the%20bots%20and%20tailors%20resumes%20in%2030%20seconds.%20Game%20changer%20for%20job%20hunting!" 
                           target="_blank" 
                           style="text-decoration: none; position: relative; display: inline-block;"
                           onmouseover="showTooltip(this, 'Share the secret to beating ATS systems with friends and family!')"
                           onmouseout="hideTooltip(this)">
                            <img src="https://upload.wikimedia.org/wikipedia/commons/0/05/Facebook_Logo_%282019%29.png" 
                                 style="width:40px; height:40px; transition: transform 0.3s ease, filter 0.3s ease;"
                                 onmouseover="this.style.transform='scale(1.2)'; this.style.filter='brightness(1.2)';"
                                 onmouseout="this.style.transform='scale(1)'; this.style.filter='brightness(1)';">
                        </a>
                        
                        <!-- Reddit -->
                        <a href="https://www.reddit.com/submit?url=https://resumewhip.com&title=Found%20an%20AI%20tool%20that%20actually%20gets%20resumes%20past%20ATS%20bots%20🤖➡️👤&text=Tired%20of%20sending%20resumes%20into%20the%20void?%20ResumeWhip%20uses%20AI%20to%20tailor%20resumes%20to%20job%20postings%20and%20beat%20ATS%20systems.%20Finally%20getting%20responses!" 
                           target="_blank"
                           style="text-decoration: none; position: relative; display: inline-block;"
                           onmouseover="showTooltip(this, 'Help Your Reddit friends beat ATS systems and land their dream jobs!')"
                           onmouseout="hideTooltip(this)">
                            <img src="https://cdn.simpleicons.org/reddit/FF4500" 
                                 style="width:40px; height:40px; transition: transform 0.3s ease, filter 0.3s ease;"
                                 onmouseover="this.style.transform='scale(1.2)'; this.style.filter='brightness(1.2)';"
                                 onmouseout="this.style.transform='scale(1)'; this.style.filter='brightness(1)';">
                        </a>
                        
                    </div>
                
                <!-- JavaScript for clean hover tooltips -->
                <script>
                function showTooltip(element, message) {
                    // Remove any existing tooltip
                    hideTooltip(element);
                    
                    // Create tooltip
                    const tooltip = document.createElement('div');
                    tooltip.className = 'custom-tooltip';
                    tooltip.innerHTML = message;
                    tooltip.style.cssText = `
                        position: absolute;
                        bottom: 50px;
                        left: 50%;
                        transform: translateX(-50%);
                        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                        color: white;
                        padding: 8px 12px;
                        border-radius: 6px;
                        font-size: 0.8em;
                        font-weight: bold;
                        white-space: nowrap;
                        z-index: 1000;
                        box-shadow: 0 4px 8px rgba(0,0,0,0.2);
                        animation: fadeIn 0.3s ease;
                    `;
                    
                    element.appendChild(tooltip);
                }
                
                function hideTooltip(element) {
                    const existing = element.querySelector('.custom-tooltip');
                    if (existing) {
                        existing.remove();
                    }
                }
                
                // Add CSS for fade-in animation
                const style = document.createElement('style');
                style.textContent = `
                    @keyframes fadeIn {
                        from { opacity: 0; transform: translateX(-50%) translateY(5px); }
                        to { opacity: 1; transform: translateX(-50%) translateY(0); }
                    }
                `;
                document.head.appendChild(style);
                </script>
                """)

        # Main content
        with gr.Column(scale=5):
            # Input section
            with gr.Row():
                resume_input = gr.File(
                    label="📄 Upload Resume Here", 
                    file_types=[".pdf", ".docx", ".md", ".txt"]
                )
                company_input = gr.Textbox(
                    label="🏢 Enter the Company's Name", 
                    placeholder="e.g., Google, Microsoft"
                )
            
            job_input = gr.Textbox(
                label="📋 Job Description (Please Paste Full Text)", 
                lines=6,
                placeholder="Paste the complete job posting here..."
            )
            
            # Credit counter
            resume_counter = gr.Markdown("### Free Resumes Left: 3")

            # Access granter
            with gr.Accordion("My Admin Access", open=False, visible=False):
                grant_access_btn = gr.Button("🟢 Grant Unlimited Access", variant="secondary")
                access_status = gr.Markdown()

            # Tools Available
            # gr.Markdown("<h2 style='text-align:center; color:#ff7f50;'>🧰 Tools In the Toolkit</h2>")
            gr.HTML("""
            <div style="text-align: center; margin: 30px 0;">
                <h2 style="font-size: 2.2em; color: #495057; font-weight: 700; margin-bottom: 15px;">
                    🧰 Tools In the Toolkit
                </h2>
                <p style="font-size: 1.1em; color: #6c757d; margin-bottom: 30px;">
                    Choose your tool below - each one gives you an edge in today's job market
                </p>
                
                <!-- TOOL NAVIGATION BUTTONS -->
                <div style="display: flex; justify-content: center; gap: 20px; flex-wrap: wrap; margin: 30px 0;">
                    <button onclick="switchToGradioTab('validator')" style="
                        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                        color: white;
                        border: none;
                        font-weight: 800;
                        font-size: 1.4em;
                        border-radius: 12px;
                        padding: 25px 40px;
                        transition: all 0.4s ease;
                        text-transform: uppercase;
                        letter-spacing: 1px;
                        min-width: 240px;
                        cursor: pointer;
                        box-shadow: 0 6px 20px rgba(102, 126, 234, 0.3);
                        animation: subtle-pulse 4s ease-in-out infinite;
                    " onmouseover="
                        this.style.transform='translateY(-6px) scale(1.03)';
                        this.style.boxShadow='0 15px 35px rgba(102, 126, 234, 0.5)';
                        this.style.background='linear-gradient(135deg, #5a67d8 0%, #667eea 100%)';
                    " onmouseout="
                        this.style.transform='translateY(0px) scale(1)';
                        this.style.boxShadow='0 6px 20px rgba(102, 126, 234, 0.3)';
                        this.style.background='linear-gradient(135deg, #667eea 0%, #764ba2 100%)';
                    ">
                        ✅ JOB VALIDATOR
                    </button>
                    
                    <button onclick="switchToGradioTab('optimizer')" style="
                        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                        color: white;
                        border: none;
                        font-weight: 800;
                        font-size: 1.4em;
                        border-radius: 12px;
                        padding: 25px 40px;
                        transition: all 0.4s ease;
                        text-transform: uppercase;
                        letter-spacing: 1px;
                        min-width: 240px;
                        cursor: pointer;
                        box-shadow: 0 6px 20px rgba(102, 126, 234, 0.3);
                        animation: subtle-pulse 4s ease-in-out infinite;
                    " onmouseover="
                        this.style.transform='translateY(-6px) scale(1.03)';
                        this.style.boxShadow='0 15px 35px rgba(102, 126, 234, 0.5)';
                        this.style.background='linear-gradient(135deg, #5a67d8 0%, #667eea 100%)';
                    " onmouseout="
                        this.style.transform='translateY(0px) scale(1)';
                        this.style.boxShadow='0 6px 20px rgba(102, 126, 234, 0.3)';
                        this.style.background='linear-gradient(135deg, #667eea 0%, #764ba2 100%)';
                    ">
                        🎯 RESUME OPTIMIZER
                    </button>
                    
                    <button onclick="switchToGradioTab('cover')" style="
                        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                        color: white;
                        border: none;
                        font-weight: 800;
                        font-size: 1.4em;
                        border-radius: 12px;
                        padding: 25px 40px;
                        transition: all 0.4s ease;
                        text-transform: uppercase;
                        letter-spacing: 1px;
                        min-width: 240px;
                        cursor: pointer;
                        box-shadow: 0 6px 20px rgba(102, 126, 234, 0.3);
                        animation: subtle-pulse 4s ease-in-out infinite;
                    " onmouseover="
                        this.style.transform='translateY(-6px) scale(1.03)';
                        this.style.boxShadow='0 15px 35px rgba(102, 126, 234, 0.5)';
                        this.style.background='linear-gradient(135deg, #5a67d8 0%, #667eea 100%)';
                    " onmouseout="
                        this.style.transform='translateY(0px) scale(1)';
                        this.style.boxShadow='0 6px 20px rgba(102, 126, 234, 0.3)';
                        this.style.background='linear-gradient(135deg, #667eea 0%, #764ba2 100%)';
                    ">
                        📝 COVER LETTER WRITER
                    </button>
                </div>
            </div>

            <style>
            @keyframes subtle-pulse {
                0%, 100% { box-shadow: 0 6px 20px rgba(102, 126, 234, 0.3) !important; }
                50% { box-shadow: 0 10px 30px rgba(102, 126, 234, 0.4) !important; }
            }

            /* Hide the default Gradio tabs since we're replacing them with buttons */
            .gradio-container .tabs .tab-nav,
            .gradio-container .tabitem .tab-nav,
            div[data-testid="tabs"] .tab-nav,
            .tab-nav {
                display: none !important;
            }

            /* Make tab content areas more prominent when active */
            .gradio-container .tab-pane.active,
            .gradio-container .tabitem.active {
                background: #f8f9fa !important;
                border: 2px solid #dee2e6 !important;
                border-radius: 15px !important;
                padding: 30px !important;
                margin-top: 20px !important;
                animation: fadeIn 0.5s ease-in-out !important;
            }

            @keyframes fadeIn {
                from { opacity: 0; transform: translateY(20px); }
                to { opacity: 1; transform: translateY(0); }
            }

            /* Style the action buttons to match */
            button:contains("Whip Up") {
                background: linear-gradient(135deg, #e74c3c 0%, #c0392b 100%) !important;
                color: white !important;
                font-size: 1.3em !important;
                font-weight: 700 !important;
                padding: 20px 40px !important;
                border-radius: 12px !important;
                border: none !important;
                width: 100% !important;
                margin-top: 20px !important;
            }

            @media (max-width: 768px) {
                button[onclick*="switchToGradioTab"] {
                    font-size: 1.2em !important;
                    padding: 20px 30px !important;
                    min-width: 200px !important;
                    margin: 8px !important;
                }
            }
            </style>

            <script>
            function switchToGradioTab(tabName) {
                // Reset all tool buttons to default state
                document.querySelectorAll('button[onclick*="switchToGradioTab"]').forEach(btn => {
                    btn.style.background = 'linear-gradient(135deg, #667eea 0%, #764ba2 100%)';
                    btn.style.transform = 'translateY(0px) scale(1)';
                    btn.style.boxShadow = '0 6px 20px rgba(102, 126, 234, 0.3)';
                });
                
                // Highlight the clicked button
                event.target.style.background = 'linear-gradient(135deg, #ff7f50 0%, #ff6b35 100%)';
                event.target.style.boxShadow = '0 8px 25px rgba(255, 127, 80, 0.6)';
                event.target.style.transform = 'translateY(-3px)';
                
                // Try multiple methods to switch Gradio tabs
                setTimeout(() => {
                    // Method 1: Try clicking the actual Gradio tab button
                    const gradioTabs = document.querySelectorAll('button[role="tab"]');
                    gradioTabs.forEach(tab => {
                        if (tab.textContent.toLowerCase().includes(tabName) || 
                            tab.id.toLowerCase().includes(tabName)) {
                            tab.click();
                        }
                    });
                    
                    // Method 2: Try finding by data attributes
                    const tabButtons = document.querySelectorAll('[data-bs-target*="' + tabName + '"]');
                    if (tabButtons.length > 0) {
                        tabButtons[0].click();
                    }
                    
                    // Method 3: Dispatch custom event for Gradio
                    window.dispatchEvent(new CustomEvent('gradio:tab-switch', {
                        detail: { tab: tabName }
                    }));
                }, 100);
                
                // Scroll to the content area
                setTimeout(() => {
                    const tabsContainer = document.querySelector('.gradio-container .tabs') || 
                                        document.querySelector('[data-testid="tabs"]');
                    if (tabsContainer) {
                        tabsContainer.scrollIntoView({
                            behavior: 'smooth',
                            block: 'start'
                        });
                    }
                }, 200);
            }

            // Initialize first button as active on page load
            document.addEventListener('DOMContentLoaded', function() {
                const firstButton = document.querySelector('button[onclick*="switchToGradioTab"]');
                if (firstButton) {
                    firstButton.style.background = 'linear-gradient(135deg, #ff7f50 0%, #ff6b35 100%)';
                    firstButton.style.boxShadow = '0 8px 25px rgba(255, 127, 80, 0.6)';
                    firstButton.style.transform = 'translateY(-3px)';
                }
            });
            </script>
            """)

            # Tools tabs
            with gr.Tabs():
                with gr.TabItem("✅ Job Validator", id="validator_tab"):
                    with gr.Row():
                        jd_date = gr.Textbox(
                            label="📅 Posting Date (Best Guess, Anyway.)", 
                            placeholder="YYYY-MM-DD (e.g., 2024-12-01)"
                        )
                        jd_title = gr.Textbox(
                            label="💼 Job Title", 
                            placeholder="e.g., Data Scientist"
                        )
                    
                    validate_btn = gr.Button("🤖 Whip Up the Job Validator", variant="primary")
                    validation_output = gr.Markdown()

                with gr.TabItem("🎯 Resume Optimizer", id="optimizer_tab"):
                    run_resume = gr.Button("🪄 Whip Up Some Resume Magic!", variant="primary")
                    resume_md = gr.Markdown(label="Preview")
                    resume_edit = gr.Textbox(
                        label="✏️ Edit Your Resume Here (optional)", 
                        lines=15
                    )
                    suggestions = gr.Markdown(label="Suggestions & Tips")
                    
                    with gr.Row():
                        export_resume_btn = gr.Button("Download PDF ➡️")
                        export_resume_result = gr.File()

                with gr.TabItem("📝 Cover Letter Writer", id="cover_tab"):
                    run_cover = gr.Button("📝 Whip Up My Cover Letter", variant="primary")
                    cover_output = gr.Textbox(
                        label="Here's Your Cover Letter. Edit To Give It Your Voice.", 
                        lines=15
                    )
                    
                    with gr.Row():
                        export_cover_btn = gr.Button("Download PDF ➡️")
                        export_cover_result = gr.File()
                
                gr.HTML("""
                    <div style="
                        text-align: center;
                        margin: 20px 0;
                        padding: 15px;
                        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                        border-radius: 15px;
                        color: white;
                        font-size: 1.1em;
                        font-weight: 600;
                        border: 3px solid #fff;
                        box-shadow: 0 8px 20px rgba(102, 126, 234, 0.3);
                    ">
                        ⚡ Pro Tip: Start with the Job Validator to check if the posting is legit, then optimize your resume! ⚡
                    </div>
                    """)  

    # Footer for SEO
    gr.HTML("""
            <footer style = "margin-top: 50px; padding: 30px 20px; border-top: 2px solid #eee;
                            background: linear-gradient(135deg, #f8f9a 0%, #e9ecef 100%);")
            
                <!-- Main footer content -->
                <div style="max-width: 1200px; margin: 0 auto; text-align: center;">
            
                    <!-- Key links -->
                    <div style="margin-bottom: 25px;">
                        <a href="mailto:support@resumewhip.com"
                            style="margin: 0 20px; color: #1e90ff; text-decoration: none; font-weight: 500;">
                            📧 Contact Support
                        </a>
                        <a href="https://buy.stripe.com/cNi9ASgWl6C614l3Ja1Jm00"
                            style="margin: 0 20px; color: #635BFF; text-decoration: none; font-weight: 500;">
                            💳 Upgrade to Premium
                        </a>
                        <a href="https://resumewhip.com/blog"
                            style="margin: 0 20px; color: #1e90ff; text-decoration: none; font-weight: 500;">
                            📖 Resume Tips Blog
                        </a>
                    </div>
            
                    <!-- Value proposition -->
                    <div style="margin-bottom: 20px; color: #495057;">
                        <h3 style="color: #343a40; margin-bottom: 10px;">
                            🏎️💨 ResumeWhip - The AI Resume Optimizer That Actually Works
                        </h3>
                        <p style="font-size: 1.1em; line-height: 1.4; max-width: 600px; margin: 0 auto;">
                            <span class="bold-text">About Us:</span> We were job seekers proficient in our fields and great with people,
                            but also job seekers who had an extremely hard time getting past ATS Systems. 
                            So, we created a resume optimizer that consistently outperforms 
                            Premium Job Platform Services by over 40%. 
                            <span class="bold-text">We're in the business of getting people past machines.</span>
                        </p>
                    </div>
            
                    <!-- Trust signals -->
                    <div style="display: flex; justify-content: center; gap: 30px; margin-bottom: 25px;
                                flex-wrap: wrap; color: #6c757d; font-size: 0.9em;">
                                        <div>🛡️ We never share or sell your data</div>
                                        <div>⚡ 30-second resume optimization</div>
                                        <div>🎯 40% higher ATS compatibility</div>
                                        <div>💼 Works with all job boards</div>
                    </div>

                    <!-- Copyright and keywords -->
                    <div style="border-top: 1px solid #dee2e6; padding-top: 20px;">
                        <p style="color: #6c757d; font-size: 0.95em; margin-bottom: 10px;">
                            © 2025 ResumeWhip - AI-Powered Resume Optimization Tool
                        </p>
                        <p style="color: #adb5bd; font-size: 0.8em; line-height: 1.3;">
                            Keywords: AI resume optimizer, ATS resume checker, resume builder, cover letter generator, 
                            job application tool, interview preparation, career advancement, resume writing service
                        </p>
                    </div>
                
                </div>
            </footer>
            """)
    # Event handlers
    validate_btn.click(
        fn=validate_job_posting,
        inputs=[job_input, jd_date, company_input, jd_title],
        outputs=validation_output
    )
    
    run_resume.click(
        fn=run_resume_with_credits,
        inputs=[resume_input, job_input],
        outputs=[resume_md, resume_edit, suggestions, resume_counter]
    )
    
    export_resume_btn.click(
        fn=export_resume,
        inputs=[resume_edit, company_input],
        outputs=export_resume_result
    )
    
    run_cover.click(
        fn=generate_cover_letter,
        inputs=[resume_input, job_input],
        outputs=cover_output
    )
    
    export_cover_btn.click(
        fn=save_cover_letter,
        inputs=[cover_output, company_input],
        outputs=export_cover_result
    )

    # # Admin event handler (for testing)
    # admin_input = gr.Textbox(label="User Email or ID")

# =========================================================================
# Add the code below for free access to users for testing (like Mia, etc.)
    # grant_access_btn.click(
    #     fn=admin_grant_access,
    #     inputs=[admin_input],
    #     outputs=access_status
    # )
# =========================================================================

